In [1]:
%pip install scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
PROJECT_ROOT = Path(
    r"C:\Users\sarun\OneDrive\Desktop\cesppl-internship"
)

PROCESSED_DATASET = (
    PROJECT_ROOT /
    "data" /
    "cesppl_processed"
)

OUTPUT_DIR = PROJECT_ROOT / "Week-09"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV_PATH = OUTPUT_DIR / "train.csv"
VAL_CSV_PATH = OUTPUT_DIR / "val.csv"
TEST_CSV_PATH = OUTPUT_DIR / "test.csv"

DATA_CARD_PATH = OUTPUT_DIR / "DATA_CARD.md"

print("Project root       :", PROJECT_ROOT)
print("Processed dataset  :", PROCESSED_DATASET)
print("Output directory   :", OUTPUT_DIR)

Project root       : C:\Users\sarun\OneDrive\Desktop\cesppl-internship
Processed dataset  : C:\Users\sarun\OneDrive\Desktop\cesppl-internship\data\cesppl_processed
Output directory   : C:\Users\sarun\OneDrive\Desktop\cesppl-internship\Week-09


In [4]:
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder was not found:\n{PROJECT_ROOT}"
    )

if not PROCESSED_DATASET.exists():
    raise FileNotFoundError(
        f"Processed dataset was not found:\n{PROCESSED_DATASET}"
    )

class_folders = sorted(
    folder
    for folder in PROCESSED_DATASET.iterdir()
    if folder.is_dir()
)

if not class_folders:
    raise ValueError(
        "No class folders were found inside the processed dataset."
    )

print("Processed dataset found successfully.")
print("Number of class folders:", len(class_folders))

for folder in class_folders:
    print("-", folder.name)

Processed dataset found successfully.
Number of class folders: 10
- BIN LIFTING
- BIN WASHING
- GATE MEETING
- LFC
- MANUAL BEACH CLEANING
- MECHANICAL SWEEPING
- MECHANIZED BEACH CLEANING
- PRIMARY COLLECTION
- ROAD SWEEPING
- SECONDARY VEHICLES


In [6]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

dataset_rows = []

for class_folder in class_folders:

    class_name = (
        class_folder.name
        .strip()
        .upper()
        .replace(" ", "_")
    )

    image_paths = sorted(
        path
        for path in class_folder.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    )

    for image_path in image_paths:

        relative_filename = image_path.relative_to(
            PROCESSED_DATASET
        )

        dataset_rows.append({
            "filename": relative_filename.as_posix(),
            "class": class_name
        })

dataset_df = pd.DataFrame(
    dataset_rows,
    columns=["filename", "class"]
)

print("Total processed images:", len(dataset_df))
print("Number of classes      :", dataset_df["class"].nunique())

display(dataset_df.head(10))

Total processed images: 3616
Number of classes      : 10


,filename,class
0,BIN LIFTING/image744__png.jpg,BIN_LIFTING
1,BIN LIFTING/image745__png.jpg,BIN_LIFTING
2,BIN LIFTING/image746__png.jpg,BIN_LIFTING
3,BIN LIFTING/image747__png.jpg,BIN_LIFTING
4,BIN LIFTING/image748__png.jpg,BIN_LIFTING
5,BIN LIFTING/image749__png.jpg,BIN_LIFTING
6,BIN LIFTING/image750__png.jpg,BIN_LIFTING
7,BIN LIFTING/image751__png.jpg,BIN_LIFTING
8,BIN LIFTING/image752__png.jpg,BIN_LIFTING
9,BIN LIFTING/image753__png.jpg,BIN_LIFTING


In [7]:
if dataset_df.empty:
    raise ValueError(
        "The dataset DataFrame is empty."
    )

if dataset_df["filename"].duplicated().any():
    duplicated_files = dataset_df[
        dataset_df["filename"].duplicated(
            keep=False
        )
    ]

    display(duplicated_files)

    raise ValueError(
        "Duplicate filenames were found in the dataset DataFrame."
    )

missing_values = dataset_df.isna().sum()

print("Missing values:")
display(missing_values.to_frame("count"))

print(
    "Duplicate filename rows:",
    dataset_df["filename"].duplicated().sum()
)

print("\nDataset validation completed successfully.")

Missing values:


,count
filename,0
class,0


Duplicate filename rows: 0

Dataset validation completed successfully.


In [8]:
class_counts_df = (
    dataset_df["class"]
    .value_counts()
    .sort_index()
    .rename_axis("class")
    .reset_index(name="total_images")
)

display(class_counts_df)

print(
    "Largest class count :",
    class_counts_df["total_images"].max()
)

print(
    "Smallest class count:",
    class_counts_df["total_images"].min()
)

imbalance_ratio = (
    class_counts_df["total_images"].max()
    /
    class_counts_df["total_images"].min()
)

print(
    "Class imbalance ratio:",
    round(imbalance_ratio, 2),
    "to 1"
)

,class,total_images
0,BIN_LIFTING,166
1,BIN_WASHING,325
2,GATE_MEETING,360
3,LFC,135
4,MANUAL_BEACH_CLEANING,1328
5,MECHANICAL_SWEEPING,179
6,MECHANIZED_BEACH_CLEANING,204
7,PRIMARY_COLLECTION,109
8,ROAD_SWEEPING,518
9,SECONDARY_VEHICLES,292


Largest class count : 1328
Smallest class count: 109
Class imbalance ratio: 12.18 to 1


In [9]:
train_val_df, test_df = train_test_split(
    dataset_df,
    test_size=0.15,
    random_state=42,
    stratify=dataset_df["class"],
    shuffle=True
)

print("Train + validation images:", len(train_val_df))
print("Test images              :", len(test_df))

Train + validation images: 3073
Test images              : 543


In [10]:
VALIDATION_FRACTION_OF_TRAIN_VAL = 15 / 85

train_df, val_df = train_test_split(
    train_val_df,
    test_size=VALIDATION_FRACTION_OF_TRAIN_VAL,
    random_state=42,
    stratify=train_val_df["class"],
    shuffle=True
)

print("Train images     :", len(train_df))
print("Validation images:", len(val_df))
print("Test images      :", len(test_df))

print(
    "Total images:",
    len(train_df) + len(val_df) + len(test_df)
)

Train images     : 2530
Validation images: 543
Test images      : 543
Total images: 3616


In [11]:
train_df = (
    train_df
    .sort_values(["class", "filename"])
    .reset_index(drop=True)
)

val_df = (
    val_df
    .sort_values(["class", "filename"])
    .reset_index(drop=True)
)

test_df = (
    test_df
    .sort_values(["class", "filename"])
    .reset_index(drop=True)
)

print("Split DataFrames sorted successfully.")

Split DataFrames sorted successfully.


In [12]:
all_classes = set(dataset_df["class"].unique())

train_classes = set(train_df["class"].unique())
val_classes = set(val_df["class"].unique())
test_classes = set(test_df["class"].unique())

missing_from_train = all_classes - train_classes
missing_from_val = all_classes - val_classes
missing_from_test = all_classes - test_classes

print("Missing from train     :", missing_from_train)
print("Missing from validation:", missing_from_val)
print("Missing from test      :", missing_from_test)

if missing_from_train:
    raise ValueError(
        f"Classes missing from training split: {missing_from_train}"
    )

if missing_from_val:
    raise ValueError(
        f"Classes missing from validation split: {missing_from_val}"
    )

if missing_from_test:
    raise ValueError(
        f"Classes missing from test split: {missing_from_test}"
    )

print("\n✅ Every class appears in every split.")

Missing from train     : set()
Missing from validation: set()
Missing from test      : set()

✅ Every class appears in every split.


In [13]:
train_counts = train_df["class"].value_counts()
val_counts = val_df["class"].value_counts()
test_counts = test_df["class"].value_counts()
total_counts = dataset_df["class"].value_counts()

split_distribution_df = pd.DataFrame({
    "total": total_counts,
    "train": train_counts,
    "validation": val_counts,
    "test": test_counts
}).fillna(0).astype(int)

split_distribution_df = (
    split_distribution_df
    .sort_index()
    .reset_index()
    .rename(columns={"index": "class"})
)

display(split_distribution_df)

,class,total,train,validation,test
0,BIN_LIFTING,166,116,25,25
1,BIN_WASHING,325,227,49,49
2,GATE_MEETING,360,252,54,54
3,LFC,135,95,20,20
4,MANUAL_BEACH_CLEANING,1328,930,199,199
5,MECHANICAL_SWEEPING,179,125,27,27
6,MECHANIZED_BEACH_CLEANING,204,142,31,31
7,PRIMARY_COLLECTION,109,77,16,16
8,ROAD_SWEEPING,518,362,78,78
9,SECONDARY_VEHICLES,292,204,44,44


In [14]:
minimum_train_count = (
    split_distribution_df["train"].min()
)

minimum_val_count = (
    split_distribution_df["validation"].min()
)

minimum_test_count = (
    split_distribution_df["test"].min()
)

print("Smallest training class count  :", minimum_train_count)
print("Smallest validation class count:", minimum_val_count)
print("Smallest test class count      :", minimum_test_count)

smallest_test_classes = split_distribution_df[
    split_distribution_df["test"]
    == minimum_test_count
]

print("\nClass or classes with the smallest test count:")
display(smallest_test_classes)

Smallest training class count  : 77
Smallest validation class count: 16
Smallest test class count      : 16

Class or classes with the smallest test count:


,class,total,train,validation,test
7,PRIMARY_COLLECTION,109,77,16,16


In [15]:
train_files = set(train_df["filename"])
val_files = set(val_df["filename"])
test_files = set(test_df["filename"])

train_val_overlap = train_files & val_files
train_test_overlap = train_files & test_files
val_test_overlap = val_files & test_files

print(
    "Train-validation overlap:",
    len(train_val_overlap)
)

print(
    "Train-test overlap:",
    len(train_test_overlap)
)

print(
    "Validation-test overlap:",
    len(val_test_overlap)
)

if train_val_overlap:
    raise ValueError(
        "Some files appear in both train and validation."
    )

if train_test_overlap:
    raise ValueError(
        "Some files appear in both train and test."
    )

if val_test_overlap:
    raise ValueError(
        "Some files appear in both validation and test."
    )

print("\n✅ No image appears in more than one split.")

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

✅ No image appears in more than one split.


In [16]:
combined_split_files = (
    train_files |
    val_files |
    test_files
)

original_files = set(dataset_df["filename"])

missing_files = original_files - combined_split_files
unexpected_files = combined_split_files - original_files

print("Original dataset files:", len(original_files))
print("Combined split files  :", len(combined_split_files))
print("Missing files         :", len(missing_files))
print("Unexpected files      :", len(unexpected_files))

if missing_files:
    raise ValueError(
        "Some processed images were not included in any split."
    )

if unexpected_files:
    raise ValueError(
        "The split files contain unknown filenames."
    )

print("\n✅ Every processed image appears in exactly one split.")

Original dataset files: 3616
Combined split files  : 3616
Missing files         : 0
Unexpected files      : 0

✅ Every processed image appears in exactly one split.


In [17]:
total_images = len(dataset_df)

split_summary_df = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "test"
    ],
    "images": [
        len(train_df),
        len(val_df),
        len(test_df)
    ]
})

split_summary_df["percentage"] = (
    split_summary_df["images"]
    / total_images
    * 100
).round(2)

display(split_summary_df)

,split,images,percentage
0,train,2530,69.97
1,validation,543,15.02
2,test,543,15.02


In [18]:
train_df.to_csv(
    TRAIN_CSV_PATH,
    index=False,
    encoding="utf-8"
)

val_df.to_csv(
    VAL_CSV_PATH,
    index=False,
    encoding="utf-8"
)

test_df.to_csv(
    TEST_CSV_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:", TRAIN_CSV_PATH)
print("Saved:", VAL_CSV_PATH)
print("Saved:", TEST_CSV_PATH)

Saved: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\Week-09\train.csv
Saved: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\Week-09\val.csv
Saved: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\Week-09\test.csv


In [19]:
saved_train_df = pd.read_csv(TRAIN_CSV_PATH)
saved_val_df = pd.read_csv(VAL_CSV_PATH)
saved_test_df = pd.read_csv(TEST_CSV_PATH)

print("Saved train rows     :", len(saved_train_df))
print("Saved validation rows:", len(saved_val_df))
print("Saved test rows      :", len(saved_test_df))

assert len(saved_train_df) == len(train_df)
assert len(saved_val_df) == len(val_df)
assert len(saved_test_df) == len(test_df)

assert list(saved_train_df.columns) == [
    "filename",
    "class"
]

assert list(saved_val_df.columns) == [
    "filename",
    "class"
]

assert list(saved_test_df.columns) == [
    "filename",
    "class"
]

print("\n✅ Split CSV files were saved and verified.")

Saved train rows     : 2530
Saved validation rows: 543
Saved test rows      : 543

✅ Split CSV files were saved and verified.


## Test-set granularity

Some of the smaller classes contain only about 19 or 20 test images.

For a class with 20 test images, one correctly or incorrectly classified image changes the class recall by approximately:

\[
\frac{1}{20} \times 100 = 5\%
\]

Therefore, per-class recall, precision, and F1-score for the smaller classes may change noticeably because of only one or two images. These metrics should be interpreted together with the confusion matrix and the number of test samples rather than as perfectly stable estimates.

In [20]:
COLLECTION_PERIOD = (
    "Collected during CESPPL field operations; "
    "exact collection dates should be confirmed "
    "with the operations team."
)

COLLECTION_LOCATIONS = (
    "CESPPL operational locations, including roads, "
    "beaches, collection points, depots, gates and "
    "waste-management activity areas."
)

DATASET_PURPOSE = (
    "To support the development and evaluation of "
    "computer-vision models that classify CESPPL "
    "operational activities from field photographs."
)

print("Data-card information prepared.")

Data-card information prepared.


In [21]:
class_count_lines = []

for _, row in class_counts_df.iterrows():
    class_count_lines.append(
        f"- **{row['class']}**: "
        f"{int(row['total_images'])} images"
    )

class_counts_markdown = "\n".join(
    class_count_lines
)

largest_class = class_counts_df.loc[
    class_counts_df["total_images"].idxmax()
]

smallest_class = class_counts_df.loc[
    class_counts_df["total_images"].idxmin()
]

data_card_text = f"""# CESPPL Dataset Card

## Dataset Summary

The processed CESPPL dataset contains **{len(dataset_df)} images**
across **{dataset_df["class"].nunique()} operational activity classes**.

The frozen dataset split uses:

- Training images: **{len(train_df)}**
- Validation images: **{len(val_df)}**
- Test images: **{len(test_df)}**
- Random seed: **42**
- Split strategy: **stratified 70/15/15 split**

## Purpose

{DATASET_PURPOSE}

The dataset is intended to help study whether operational activities
can be distinguished using image-classification methods.

## Dataset Content

The dataset contains the following classes:

{class_counts_markdown}

All processed images were corrected for EXIF orientation, converted to
RGB, resized and centre-cropped to **320 × 320 pixels**, and saved as
JPEG images.

Identical perceptual-hash duplicates were removed before creating the
train, validation and test splits.

## Collection Process

The images were captured by field staff using mobile phones during
real CESPPL operational activities.

**Collection period:** {COLLECTION_PERIOD}

**Collection locations:** {COLLECTION_LOCATIONS}

The dataset includes photographs of roads, beaches, vehicles, bins,
workers, depots, gates and waste-management environments.

## Labelling Process

The images were labelled through folder placement by the CESPPL
operations team.

Each folder represents one operational activity. The folder name is
used as the class label.

The labels were visually reviewed during Week 9 to identify possible
mislabelled, unclear, low-quality or multi-activity images.

## Dataset Splits

The dataset was split once using
`sklearn.model_selection.train_test_split`.

The split was stratified by class and used `random_state=42`.

The resulting files are:

- `train.csv`
- `val.csv`
- `test.csv`

These split files are frozen and should be reused for all future
experiments. They should not be regenerated between model runs.

## Class Balance

The largest class is **{largest_class["class"]}** with
**{int(largest_class["total_images"])} images.

The smallest class is **{smallest_class["class"]}** with
**{int(smallest_class["total_images"])} images.

The largest-to-smallest class ratio is approximately
**{imbalance_ratio:.2f}:1**.

This imbalance may cause the model to perform better on common classes
and less consistently on smaller classes.

## Known Limitations

- The dataset is imbalanced across the ten classes.
- Some images contain timestamps, GPS overlays, application overlays or
  camera watermarks.
- Overlay styles may be associated with particular phones, staff
  members or classes and could become shortcut features.
- Lighting varies between daylight, early morning, low-light and night
  conditions.
- Images may contain motion blur, unusual camera angles or partial
  views of the activity.
- Some visually similar classes share the same workers, roads, bins,
  beaches and vehicles.
- Some photographs may show more than one activity in the same frame.
- The dataset represents a specific operational environment and may not
  generalise to other organisations, cities or waste-management systems.
- Small test classes contain approximately 20 images, so one prediction
  may change class recall by about five percentage points.

## Ethical Considerations

Workers may be identifiable in some images.

The images should remain internal to the authorised project and should
not be publicly released without organisational approval and an
appropriate privacy review.

The dataset should not be used for facial recognition, worker
identification, employee monitoring or individual performance
assessment.

## Intended Uses

- Internal research on operational activity classification.
- Training and evaluating image-classification models.
- Studying class imbalance and visual similarity between operational
  activities.
- Supporting future internal waste-management automation research.

## Out-of-Scope Uses

- Facial recognition or worker identification.
- Surveillance or employee performance monitoring.
- Public release without permission.
- Use in unrelated organisations without validation.
- Safety-critical decisions without human review.
- Inferring sensitive information about workers.

## Maintenance

The train, validation and test splits are frozen.

Future experiments must read the filenames from the saved CSV files
instead of generating new random splits.
"""

DATA_CARD_PATH.write_text(
    data_card_text,
    encoding="utf-8"
)

print("Saved:", DATA_CARD_PATH)

Saved: C:\Users\sarun\OneDrive\Desktop\cesppl-internship\Week-09\DATA_CARD.md


In [22]:
print(
    DATA_CARD_PATH.read_text(
        encoding="utf-8"
    )
)

# CESPPL Dataset Card

## Dataset Summary

The processed CESPPL dataset contains **3616 images**
across **10 operational activity classes**.

The frozen dataset split uses:

- Training images: **2530**
- Validation images: **543**
- Test images: **543**
- Random seed: **42**
- Split strategy: **stratified 70/15/15 split**

## Purpose

To support the development and evaluation of computer-vision models that classify CESPPL operational activities from field photographs.

The dataset is intended to help study whether operational activities
can be distinguished using image-classification methods.

## Dataset Content

The dataset contains the following classes:

- **BIN_LIFTING**: 166 images
- **BIN_WASHING**: 325 images
- **GATE_MEETING**: 360 images
- **LFC**: 135 images
- **MANUAL_BEACH_CLEANING**: 1328 images
- **MECHANICAL_SWEEPING**: 179 images
- **MECHANIZED_BEACH_CLEANING**: 204 images
- **PRIMARY_COLLECTION**: 109 images
- **ROAD_SWEEPING**: 518 images
- **SECONDARY_VEHICLES**: 292

In [23]:
required_output_files = [
    TRAIN_CSV_PATH,
    VAL_CSV_PATH,
    TEST_CSV_PATH,
    DATA_CARD_PATH
]

print("========== Final Output Check ==========")

all_outputs_exist = True

for file_path in required_output_files:

    exists = file_path.exists()

    print(
        f"{file_path.name}:",
        "FOUND" if exists else "MISSING"
    )

    if not exists:
        all_outputs_exist = False

if not all_outputs_exist:
    raise FileNotFoundError(
        "One or more required output files are missing."
    )

print("\n✅ All required files were created successfully.")

========== Final Output Check ==========
train.csv: FOUND
val.csv: FOUND
test.csv: FOUND
DATA_CARD.md: FOUND

✅ All required files were created successfully.


# Week 9 Thursday Summary

## Completed Work

- Built a DataFrame containing one row per processed image.
- Used the columns `filename` and `class`.
- Created one stratified 70/15/15 train, validation and test split.
- Used `random_state=42` for reproducibility.
- Verified that every class appears in every split.
- Verified that the splits have no overlapping filenames.
- Verified that every processed image appears in exactly one split.
- Saved the frozen splits as `train.csv`, `val.csv` and `test.csv`.
- Documented the effect of small test-set sizes on per-class metrics.
- Created `DATA_CARD.md` describing the dataset, collection, labelling,
  limitations, ethical considerations and intended uses.

## Important Decision

These split CSV files are now frozen. All future training and evaluation
notebooks must use these same files. The dataset must not be randomly
split again between experiments.